In [21]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy import stats
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras import layers, models
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler

In [ ]:
# ---------------------- Global Parameter Configuration ----------------------
# Total number of clients participating in the federated learning system
NUM_CLIENTS = 20
# California Housing is a regression problem, output dimension is 1 (house price)
OUTPUT_DIM = 1
# Input feature dimension (California Housing has 8 features)
INPUT_DIM = 8
# Time window size for calculating client historical performance trends
WINDOW_SIZE = 3
# Weight of "historical performance trend" in client selection score
WEIGHT_HISTORY = 0.6
# Weight of "data quality (similarity to global model)" in client selection score
WEIGHT_DATA = 0.3
# Weight of "system status" in client selection score
WEIGHT_SYS = 0.1
# Number of clients selected to participate in each round of federated learning
NUM_SELECT_CLIENTS = 15
# Coefficient for smoothing current accuracy with historical average accuracy in adaptive aggregation
LAMBDA_SMOOTH = 0.9
# Number of training epochs each selected client performs locally
LOCAL_EPOCHS = 1
# Learning rate
LEARNING_RATE = 0.01
# Batch size for local training
BATCH_SIZE = 40
# Total number of communication rounds executed by the global coordinator (server)
NUM_GLOBAL_EPOCHS = 50
FIXED_RATIOS = [0.1, 0.1, 0.1, 0.1, 0.05, 0.05, 0.05, 0.1, 0.1, 0.1, 0.005, 0.005, 0.02, 0.03, 0.002, 0.003, 0.002, 0.003, 0.04, 0.04]
# Set random seeds to ensure experiment reproducibility
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# ---------------------- Load California Housing Data ----------------------
def load_california_housing_data():
    """Load and preprocess California Housing dataset"""
    # Define data file path
    data_file_path = 'C:/Users/GDPI/.keras/datasets/CaliforniaHousing/cal_housing.data'
    # Read data file
    data = np.loadtxt(data_file_path, delimiter=',', skiprows=1)
    # Split features and target variable
    X = data[:, :-1]
    y = data[:, -1]
    # Standardize features
    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(X)
    # Standardize labels (optional but usually beneficial for regression problems)
    scaler_y = StandardScaler()
    y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()
    # Split into training and test sets (80-20 split)
    split_idx = int(0.8 * len(X_scaled))
    X_train, X_test = X_scaled[:split_idx], X_scaled[split_idx:]
    y_train, y_test = y_scaled[:split_idx], y_scaled[split_idx:]
    return (X_train, y_train), (X_test, y_test), scaler_y

# Load data
(X_train, y_train), (X_test, y_test), y_scaler = load_california_housing_data()
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")

In [ ]:
# ---------------------- Utility Functions ----------------------
def normalize_scores(scores):
    min_score = np.min(scores)
    max_score = np.max(scores)
    if max_score - min_score < 1e-6:
        return np.ones_like(scores) / len(scores)
    return (scores - min_score) / (max_score - min_score)

def build_model(input_dim, output_dim):
    """Build regression model"""
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(output_dim)  # Regression problem, no activation function used
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='mse',  # Mean Squared Error (MSE)
        metrics=['mae']  # Mean Absolute Error (MAE)
    )
    return model

In [ ]:
# ---------------------- Non-IID Data Partitioning ----------------------
def load_and_partition_housing_non_iid(
    num_clients=10,
    client_ratios=None,
    alpha=0.5,
    val_ratio=0.3
     ):
    """
    Partition the California Housing training set into Non-IID client data according to specified ratios.
    Samples are randomly allocated by ratio, not by label classification.
    """
    total_samples = len(X_train)
    
    # Set default ratios (uniform distribution)
    if client_ratios is None:
        # Use Dirichlet distribution to generate non-uniform ratios
        client_ratios = np.random.dirichlet([alpha] * num_clients)
    else:
        assert len(client_ratios) == num_clients, "client_ratios length must equal num_clients"
        assert abs(sum(client_ratios) - 1.0) < 1e-6, "client_ratios sum must be 1.0"
    
    # Calculate sample size for each client
    client_sizes = [int(ratio * total_samples) for ratio in client_ratios]
    diff = total_samples - sum(client_sizes)
    
    # Distribute remainder to first diff clients
    for i in range(diff):
        client_sizes[i] += 1
    
    # Create indices for all samples
    all_indices = np.arange(total_samples)
    np.random.shuffle(all_indices)
    
    # Assign samples to each client
    partitions = []
    start = 0
    for size in client_sizes:
        end = start + size
        indices = all_indices[start:end]
        partitions.append(indices)
        start = end
    
    client_data = []
    
    for i in range(num_clients):
        selected = partitions[i]
        
        # Shuffle order
        np.random.shuffle(selected)
        
        X_local = X_train[selected]
        y_local = y_train[selected]
        
        # Split into training/validation sets
        split = int((1 - val_ratio) * len(X_local))
        
        client_data.append({
            'X_train': X_local[:split],
            'y_train': y_local[:split],
            'X_val': X_local[split:],
            'y_val': y_local[split:]
        })
    
    # Print statistics
    actual_sizes = [len(d['X_train']) + len(d['X_val']) for d in client_data]
    print(f"[Proportional random partition completed] Total samples: {total_samples}")
    print(f"Client data size → min: {min(actual_sizes)}, max: {max(actual_sizes)}, mean: {np.mean(actual_sizes):.1f}")
    
    # Print label statistics for each client (optional)
    print("\nLabel distribution statistics for each client:")
    for i in range(min(10, num_clients)):  # Only show first 5 clients
        client = client_data[i]
        print(f"Client {i}: Samples={len(client['y_train'])+len(client['y_val'])}, "
              f"Label mean={np.mean(client['y_train']):.3f}±{np.std(client['y_train']):.3f}, "
              f"Label range=[{np.min(client['y_train']):.3f}, {np.max(client['y_train']):.3f}]")
    
    return client_data

In [ ]:
# ---------------------- Client Class ----------------------
class Client:
    def __init__(self, client_id, data_dict):
        self.client_id = client_id
        self.X_train = data_dict['X_train']
        self.y_train = data_dict['y_train']
        self.X_val = data_dict['X_val']
        self.y_val = data_dict['y_val']
        self.model = build_model(INPUT_DIM, OUTPUT_DIM)
        self.mse_history = []  # Store MSE history
        self.mse_trend = 0.0
        self.data_similarity = 0.0
        self.sys_score = np.random.uniform(0.7, 1.0)
    
    def local_train(self, global_weights):
        self.model.set_weights(global_weights)
        self.model.fit(self.X_train, self.y_train,
                       epochs=LOCAL_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
        return self.model.get_weights()
    
    def local_evaluate(self):
        # Use MSE for regression problems
        y_pred = self.model.predict(self.X_val, verbose=0).flatten()
        mse = mean_squared_error(self.y_val, y_pred)
        self.mse_history.append(mse)
        
        # Calculate MSE trend (negative slope indicates performance improvement)
        hist = self.mse_history
        if len(hist) >= WINDOW_SIZE:
            x = np.arange(WINDOW_SIZE)
            slope, _, _, _, _ = stats.linregress(x, hist[-WINDOW_SIZE:])
            self.mse_trend = -slope  # Negative slope indicates decreasing MSE, improving performance
        else:
            x = np.arange(len(hist))
            if len(x) >= 2:
                slope, _, _, _, _ = stats.linregress(x, hist)
                self.mse_trend = -slope
            else:
                self.mse_trend = 0.0
        
        return mse
    
    def get_historical_performance_factor(self):
        # For MSE, higher trend values are better (negative MSE trend)
        return normalize_scores(np.array([self.mse_trend]))[0]
    
    def get_data_quality_factor(self):
        return normalize_scores(np.array([self.data_similarity]))[0]
    
    def get_system_factor(self):
        return self.sys_score

In [ ]:
# ---------------------- Server: FedDAAW ----------------------
class ServerDA:
    def __init__(self):
        self.global_weights = build_model(INPUT_DIM, OUTPUT_DIM).get_weights()
        self.global_mse_history = []  # Store global MSE
    
    def calculate_selection_score(self, clients):
        scores = []
        for c in clients:
            score = (WEIGHT_HISTORY * c.get_historical_performance_factor() +
                     WEIGHT_DATA * c.get_data_quality_factor() +
                     WEIGHT_SYS * c.get_system_factor())
            scores.append(score)
        return np.array(scores)
    
    def select_clients(self, clients):
        scores = self.calculate_selection_score(clients)
        probs = normalize_scores(scores)
        probs /= probs.sum()
        indices = np.random.choice(len(clients), size=NUM_SELECT_CLIENTS, replace=False, p=probs)
        return [clients[i] for i in indices]
    
    def calculate_adaptive_weights(self, selected_clients):
        weights = []
        for c in selected_clients:
            # Use inverse of MSE as weight (lower MSE gets higher weight)
            current = c.mse_history[-1] if c.mse_history else 1.0
            # Avoid division by zero
            if current < 1e-8:
                current = 1e-8
            weight = 1.0 / current
            weights.append(weight)
        
        weights = np.array(weights)
        weights = normalize_scores(weights)
        return weights / weights.sum()
    
    def update_data_similarity(self, selected_clients, client_weights_list):
        # Calculate cosine similarity between global model and client model's first layer weights
        if len(self.global_weights) > 0:
            g0 = self.global_weights[0].flatten()
            g_norm = np.linalg.norm(g0)
            
            for c, cw in zip(selected_clients, client_weights_list):
                if len(cw) > 0:
                    c0 = cw[0].flatten()
                    c_norm = np.linalg.norm(c0)
                    
                    if g_norm < 1e-8 or c_norm < 1e-8:
                        c.data_similarity = 0.0
                    else:
                        c.data_similarity = np.dot(g0, c0) / (g_norm * c_norm)
    
    def aggregate(self, selected_clients, client_weights_list):
        adaptive_w = self.calculate_adaptive_weights(selected_clients)
        new_weights = []
        
        for layer_idx in range(len(self.global_weights)):
            weighted_sum = sum(alpha * cw[layer_idx] for alpha, cw in zip(adaptive_w, client_weights_list))
            new_weights.append(weighted_sum)
        
        self.global_weights = new_weights
        self.update_data_similarity(selected_clients, client_weights_list)
    
    def evaluate(self, clients):
        total_mse = 0.0
        total_mae = 0.0
        temp_model = build_model(INPUT_DIM, OUTPUT_DIM)
        
        for c in clients:
            temp_model.set_weights(self.global_weights)
            y_pred = temp_model.predict(c.X_val, verbose=0).flatten()
            total_mse += mean_squared_error(c.y_val, y_pred)
            total_mae += mean_absolute_error(c.y_val, y_pred)
        
        avg_mse = total_mse / len(clients)
        avg_mae = total_mae / len(clients)
        self.global_mse_history.append(avg_mse)
        
        return avg_mse, avg_mae

In [62]:
# ---------------------- Server: FedAvg (Baseline) ----------------------
class ServerFedAvg:
    def __init__(self):
        self.global_weights = build_model(INPUT_DIM, OUTPUT_DIM).get_weights()
        self.global_mse_history = []
    
    def select_clients(self, clients):
        indices = np.random.choice(len(clients), size=NUM_SELECT_CLIENTS, replace=False)
        return [clients[i] for i in indices]
    
    def aggregate(self, client_weights_list):
        K = len(client_weights_list)
        new_weights = []
        
        for layer_idx in range(len(self.global_weights)):
            avg_layer = sum(cw[layer_idx] for cw in client_weights_list) / K
            new_weights.append(avg_layer)
        
        self.global_weights = new_weights
    
    def evaluate(self, clients):
        total_mse = 0.0
        total_mae = 0.0
        temp_model = build_model(INPUT_DIM, OUTPUT_DIM)
        
        for c in clients:
            temp_model.set_weights(self.global_weights)
            y_pred = temp_model.predict(c.X_val, verbose=0).flatten()
            total_mse += mean_squared_error(c.y_val, y_pred)
            total_mae += mean_absolute_error(c.y_val, y_pred)
        
        avg_mse = total_mse / len(clients)
        avg_mae = total_mae / len(clients)
        self.global_mse_history.append(avg_mse)
        
        return avg_mse, avg_mae

In [ ]:
# ---------------------- Main Training Process: Comparative Experiment ----------------------
# Lists to store evaluation metrics per round for both methods
da_mse_history = []
da_mae_history = []
fedavg_mse_history = []
fedavg_mae_history = []

# Initialize servers for both aggregation methods
server_da = ServerDA()
server_fedavg = ServerFedAvg()

def main():
    print("Loading California Housing dataset and performing Non-IID partition...")
    
    # Generate client data ratios using Dirichlet distribution
    alpha = 0.5
    ratios = np.random.dirichlet([alpha] * NUM_CLIENTS)
    
    # Partition data
    # Example fixed ratios (commented out): [0.1, 0.1, 0.1, ...]
    client_datasets = load_and_partition_housing_non_iid(
        NUM_CLIENTS,         
        alpha=0.5,           # Dirichlet parameter controlling Non-IID degree
        client_ratios=FIXED_RATIOS,
        val_ratio=0.3
    )
    
    # Create two identical sets of clients for fair comparison
    clients_da = [Client(i, client_datasets[i]) for i in range(NUM_CLIENTS)]
    clients_fedavg = [Client(i, client_datasets[i]) for i in range(NUM_CLIENTS)]
    
    print("Starting comparative training: Dynamic-Adaptive vs FedAvg")
    
    # Main federated training loop
    for epoch in range(NUM_GLOBAL_EPOCHS):
        print(f"\n========== Global Round {epoch + 1}/{NUM_GLOBAL_EPOCHS} ==========")
        
        # ----- Dynamic-Adaptive Aggregation -----
        selected_da = server_da.select_clients(clients_da)
        weights_da = [c.local_train(server_da.global_weights) for c in selected_da]
        
        # Local evaluation on selected clients
        for c in selected_da:
            c.local_evaluate()
        
        # Server aggregation and global evaluation
        server_da.aggregate(selected_da, weights_da)
        mse_da, mae_da = server_da.evaluate(clients_da)
        da_mse_history.append(mse_da)
        da_mae_history.append(mae_da)
        
        # ----- Standard FedAvg -----
        selected_fed = server_fedavg.select_clients(clients_fedavg)
        weights_fed = [c.local_train(server_fedavg.global_weights) for c in selected_fed]
        
        for c in selected_fed:
            c.local_evaluate()
        
        server_fedavg.aggregate(weights_fed)
        mse_fed, mae_fed = server_fedavg.evaluate(clients_fedavg)
        fedavg_mse_history.append(mse_fed)
        fedavg_mae_history.append(mae_fed)
        
        # Print round results
        print(f"DA - MSE: {mse_da:.4f}, MAE: {mae_da:.4f} | FedAvg - MSE: {mse_fed:.4f}, MAE: {mae_fed:.4f}")

if __name__ == "__main__":
    main()

In [ ]:
if True:  # Visualization and final evaluation block
    # ----- Visualization Comparison -----
    rounds = range(1, NUM_GLOBAL_EPOCHS + 1)
    
    # MSE Comparison Plot
    plt.figure(figsize=(14, 6))
    
    plt.subplot(1, 2, 1)
    plt.plot(rounds, da_mse_history, 'b-o', label='FedDAAW', linewidth=2)
    plt.plot(rounds, fedavg_mse_history, 'r--s', label='FedAvg', linewidth=2)
    plt.xlabel('Training Rounds on Non-IID California Housing')
    plt.ylabel('Mean Squared Error (MSE)')
    plt.title('California Housing: MSE Comparison')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    # MAE Comparison Plot
    plt.subplot(1, 2, 2)
    plt.plot(rounds, da_mae_history, 'b-o', label='FedDAAW', linewidth=2)
    plt.plot(rounds, fedavg_mae_history, 'r--s', label='FedAvg', linewidth=2)
    plt.xlabel('Training Rounds on Non-IID California Housing')
    plt.ylabel('Mean Absolute Error (MAE)')
    plt.title('California Housing: MAE Comparison')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()
    
    # Print final results
    print("\n===== Final Results =====")
    print(f"Dynamic-Adaptive - Final MSE: {da_mse_history[-1]:.4f}, Final MAE: {da_mae_history[-1]:.4f}")
    print(f"FedAvg - Final MSE: {fedavg_mse_history[-1]:.4f}, Final MAE: {fedavg_mae_history[-1]:.4f}")
    
    # Evaluate final models on test set
    print("\n===== Test Set Evaluation =====")
    
    # Create test models
    test_model_da = build_model(INPUT_DIM, OUTPUT_DIM)
    test_model_da.set_weights(server_da.global_weights)
    
    test_model_fedavg = build_model(INPUT_DIM, OUTPUT_DIM)
    test_model_fedavg.set_weights(server_fedavg.global_weights)
    
    # Generate predictions
    y_pred_da = test_model_da.predict(X_test, verbose=0).flatten()
    y_pred_fedavg = test_model_fedavg.predict(X_test, verbose=0).flatten()
    
    # Calculate evaluation metrics
    mse_da_test = mean_squared_error(y_test, y_pred_da)
    mae_da_test = mean_absolute_error(y_test, y_pred_da)
    
    mse_fedavg_test = mean_squared_error(y_test, y_pred_fedavg)
    mae_fedavg_test = mean_absolute_error(y_test, y_pred_fedavg)
    
    print(f"Dynamic-Adaptive - Test MSE: {mse_da_test:.4f}, Test MAE: {mae_da_test:.4f}")
    print(f"FedAvg - Test MSE: {mse_fedavg_test:.4f}, Test MAE: {mae_fedavg_test:.4f}")

